# 03 — Model Evaluation

This notebook evaluates the trained YOLO model on the test set, computing IoU, Precision, Recall, F1, mAP@50, and mAP@50:95, and generating evaluation visualizations.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
from ultralytics import YOLO

from src.utils.metrics import (
    compute_iou,
    compute_precision_recall,
    compute_map,
    generate_evaluation_report,
)
from src.utils.visualization import (
    plot_precision_recall_curve,
    plot_confusion_matrix,
    plot_map_curve,
    plot_class_distribution,
)

## Configuration

In [ ]:
MODEL_PATH = '../models/best.pt'   # Path to trained model
DATA_YAML = '../configs/data.yaml'
OUTPUT_DIR = Path('../outputs/metrics')
PLOTS_DIR = Path('../outputs/plots')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: 'person', 1: 'car', 2: 'bicycle', 3: 'motorcycle'}
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.5

## Step 1 — Run YOLO Validation (Built-in Metrics)

In [ ]:
model = YOLO(MODEL_PATH)
val_results = model.val(data=DATA_YAML, split='test', conf=CONF_THRESHOLD)

print('Built-in YOLO Evaluation Results')
print('=' * 40)
print(f'mAP@50:      {val_results.box.map50:.4f}')
print(f'mAP@50-95:   {val_results.box.map:.4f}')
print(f'Precision:   {val_results.box.mp:.4f}')
print(f'Recall:      {val_results.box.mr:.4f}')

## Step 2 — Per-Class Metrics Table

In [ ]:
# Per-class results from YOLO validation
per_class_map50 = val_results.box.maps  # mAP per class at IoU=0.5

results_table = []
for i, class_name in CLASS_NAMES.items():
    if i < len(per_class_map50):
        results_table.append({
            'Class': class_name,
            'mAP@50': f'{per_class_map50[i]:.4f}',
        })

df = pd.DataFrame(results_table)
print(df.to_string(index=False))
df.to_csv(OUTPUT_DIR / 'per_class_metrics.csv', index=False)

## Step 3 — Custom IoU Analysis

Demonstrate understanding of IoU by computing it manually on sample predictions.

In [ ]:
# Example: manual IoU calculation between predicted and ground-truth boxes
example_pred = [100, 100, 300, 300]
example_gt   = [120, 110, 310, 310]

iou = compute_iou(example_pred, example_gt)
print(f'Example IoU: {iou:.4f}')
print(f'Interpretation: {"Good match" if iou > 0.5 else "Poor match"} (threshold = 0.5)')

## Step 4 — Confidence Threshold Impact Analysis

In [ ]:
# Analyze how confidence threshold affects precision/recall
import matplotlib.pyplot as plt

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
precisions_at_thresh = []
recalls_at_thresh = []

for thresh in thresholds:
    res = model.val(data=DATA_YAML, split='test', conf=thresh, verbose=False)
    precisions_at_thresh.append(res.box.mp)
    recalls_at_thresh.append(res.box.mr)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precisions_at_thresh, 'b-o', label='Precision')
ax.plot(thresholds, recalls_at_thresh, 'r-o', label='Recall')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision & Recall vs Confidence Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'threshold_analysis.png', dpi=150)
plt.show()

## Step 5 — Inference Speed

In [ ]:
import time
import cv2

test_img_dir = Path('../data/dataset/test/images')
test_images = list(test_img_dir.glob('*.[jp][pn]g'))[:20] if test_img_dir.exists() else []

if test_images:
    times = []
    for img_path in test_images:
        img = cv2.imread(str(img_path))
        start = time.time()
        model.predict(img, verbose=False)
        times.append(time.time() - start)
    
    avg_time = np.mean(times)
    fps = 1.0 / avg_time
    print(f'Average inference time: {avg_time*1000:.1f} ms')
    print(f'Inference FPS: {fps:.1f}')
else:
    print('No test images found.')

## Summary

| Metric | Value |
|--------|-------|
| mAP@50 | _fill after training_ |
| mAP@50:95 | _fill after training_ |
| Precision | _fill after training_ |
| Recall | _fill after training_ |
| FPS | _fill after training_ |